In [1]:
import duckdb
from pathlib import Path

# paths
DATA_PROCESSED = Path("../data_processed").resolve()
DATA_PROCESSED.mkdir(exist_ok=True)

out_parquet = DATA_PROCESSED / "calls_2025_full.parquet"

con = duckdb.connect()


In [2]:
API_URL = "https://data.seattle.gov/resource/33kz-ixgy.csv"


In [5]:
#sanity check for correct host connection


API_BASE = "https://cos-data.seattle.gov/resource/33kz-ixgy.csv"  # correct host
con = duckdb.connect()

# Pull 5 rows to confirm it works
df5 = con.execute(f"SELECT * FROM read_csv_auto('{API_BASE}?$limit=5')").df()
df5.head()


,cad_event_number,cad_event_clearance_description,call_type,priority,initial_call_type,final_call_type,cad_event_original_time_queued,cad_event_arrived_time,dispatch_precinct,dispatch_sector,...,call_sign_dispatch_delay_time_s_,call_sign_response_time_s_,call_sign_at_scene_time,cad_event_first_response_time_s_,call_sign_in_service_time,call_type_indicator,dispatch_neighborhood,call_type_received_classification,dispatch_address,count_of_officers
0,2017000382812,OTHER REPORT MADE,ONVIEW,7,TRAFFIC STOP - OFFICER INITIATED ONVIEW,TRAFFIC - MOVING VIOLATION,2017-10-14 21:46:20,2017-10-14 21:46:20,WEST,KING,...,0,0,2017-10-14 21:46:20,0,2017-10-14 21:54:22,ONVIEW,CHINATOWN/INTERNATIONAL DISTRICT,OFFICER_GENERATED,14XX BLOCK OF S DEARBORN ST,1
1,2010000391714,REPORT WRITTEN (NO ARREST),"TELEPHONE OTHER, NOT 911",3,THREATS (INCLS IN-PERSON/BY PHONE/IN WRITING),"ASSAULTS - HARASSMENT, THREATS",2010-11-09 14:03:11,2010-11-09 14:10:14,NORTH,UNION,...,121,423,2010-11-09 14:10:14,423,2010-11-09 15:27:51,DISPATCH,ROOSEVELT/RAVENNA,COMMUNITY_GENERATED,25XX BLOCK OF NE 54 ST,1
2,2010000392016,ASSISTANCE RENDERED,911,3,"SUSPICIOUS PERSON, VEHICLE, OR INCIDENT",SUSPICIOUS CIRCUM. - SUSPICIOUS VEHICLE,2010-11-09 18:57:35,2010-11-09 19:59:55,WEST,QUEEN,...,2791,3740,2010-11-09 19:59:55,3740,2010-11-09 20:19:18,DISPATCH,MAGNOLIA,COMMUNITY_GENERATED,45XX BLOCK OF TEXAS WY W,1
3,2014000231479,REPORT WRITTEN (NO ARREST),911,1,ORDER - CRITICAL VIOLATION OF DV COURT ORDER,DV - ASSIST VICTIM BY COURT ORDER,2014-07-15 18:48:14,2014-07-15 19:00:33,NORTH,JOHN,...,179,739,2014-07-15 19:00:33,739,2014-07-15 19:51:25,DISPATCH,GREENWOOD,COMMUNITY_GENERATED,REDACTED,2
4,2010000392024,CITATION,ONVIEW,7,TRAFFIC STOP - OFFICER INITIATED ONVIEW,TRAFFIC - MOVING VIOLATION,2010-11-09 19:10:51,2010-11-09 19:10:51,NORTH,NORA,...,0,0,2010-11-09 19:10:51,0,2010-11-09 19:43:47,ONVIEW,NORTHGATE,OFFICER_GENERATED,MERIDIAN AV N / N NORTHGATE WY,1


In [6]:
pip install requests pyarrow


Note: you may need to restart the kernel to use updated packages.


In [9]:
import time
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from io import StringIO

API_BASE = "https://cos-data.seattle.gov/resource/33kz-ixgy.csv"
TIME_COL = "cad_event_original_time_queued"

out_dir = Path("../data_processed").resolve()
out_dir.mkdir(exist_ok=True)
out_file = out_dir / "calls_2025_full.parquet"

# Try 25k first; if your connection is strong, you can bump back to 50k.
LIMIT = 25_000

# Resume support: set this to the offset where it timed out.
# You said you downloaded 4 chunks at 50k => offset was 200_000.
# If you now use LIMIT=25k, resume offset should be 200_000 (still valid).
START_OFFSET = 200_000

where = f"{TIME_COL} >= '2025-01-01T00:00:00' AND {TIME_COL} < '2026-01-01T00:00:00'"

float_cols = [
    "care_call_sign_total_service_time_s_",
    "co_response_call_sign_total_service_time_s_",
    "spd_call_sign_total_service_time_s_",
    "call_sign_total_service_time_s_",
    "first_care_call_sign_dispatch_delay_time_s_",
    "first_care_call_sign_response_time_s_",
    "first_co_response_call_sign_dispatch_delay_time_s_",
    "first_co_response_call_sign_response_time_s_",
    "first_spd_call_sign_dispatch_delay_time_s_",
    "first_spd_call_sign_response_time_s_",
    "call_sign_response_time_s_",
    "cad_event_first_response_time_s_",
]

int_cols = ["cad_event_number", "priority", "count_of_officers"]

session = requests.Session()

def fetch_csv_page(offset: int, limit: int, max_retries: int = 6, timeout: int = 240) -> str:
    params = {"$where": where, "$limit": limit, "$offset": offset}
    last_err = None
    for attempt in range(max_retries):
        try:
            r = session.get(API_BASE, params=params, timeout=timeout)
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_err = e
            sleep_s = min(60, 2 ** attempt)  # 1,2,4,8,16,32,60
            print(f"⚠️ Fetch failed at offset={offset:,} attempt={attempt+1}/{max_retries}: {e}")
            print(f"   Sleeping {sleep_s}s and retrying...")
            time.sleep(sleep_s)
    raise last_err

# If you already wrote a partial parquet and want to start fresh, delete it first.
# out_file.unlink(missing_ok=True)

writer = None
target_schema = None
offset = START_OFFSET
total_rows = 0

# If starting at 0, you can still run this — it'll create writer on first chunk.
# If resuming at nonzero, you should start fresh (recommended) OR switch to multi-file mode (below).
# Easiest: start fresh with START_OFFSET=0 (most reliable).

while True:
    csv_text = fetch_csv_page(offset, LIMIT, max_retries=6, timeout=240)
    chunk = pd.read_csv(StringIO(csv_text))

    if chunk.empty:
        break

    for c in float_cols:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce").astype("float64")

    for c in int_cols:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce").astype("Int64")

    table = pa.Table.from_pandas(chunk, preserve_index=False)

    # Create writer on first chunk
    if writer is None:
        target_schema = table.schema
        writer = pq.ParquetWriter(out_file, target_schema, compression="zstd")
    else:
        table = table.cast(target_schema)

    writer.write_table(table)

    n = len(chunk)
    total_rows += n
    offset += LIMIT
    print(f"✅ Downloaded {n:,} rows (run total {total_rows:,})... next offset={offset:,}")

if writer is not None:
    writer.close()

print("DONE:", out_file)
print("Rows downloaded this run:", f"{total_rows:,}")
print("Next offset would be:", f"{offset:,}")


✅ Downloaded 25,000 rows (run total 25,000)... next offset=225,000
✅ Downloaded 25,000 rows (run total 50,000)... next offset=250,000
✅ Downloaded 25,000 rows (run total 75,000)... next offset=275,000
✅ Downloaded 25,000 rows (run total 100,000)... next offset=300,000
✅ Downloaded 25,000 rows (run total 125,000)... next offset=325,000
✅ Downloaded 25,000 rows (run total 150,000)... next offset=350,000
✅ Downloaded 25,000 rows (run total 175,000)... next offset=375,000
✅ Downloaded 25,000 rows (run total 200,000)... next offset=400,000
✅ Downloaded 25,000 rows (run total 225,000)... next offset=425,000
✅ Downloaded 25,000 rows (run total 250,000)... next offset=450,000
✅ Downloaded 25,000 rows (run total 275,000)... next offset=475,000
✅ Downloaded 25,000 rows (run total 300,000)... next offset=500,000
✅ Downloaded 25,000 rows (run total 325,000)... next offset=525,000
✅ Downloaded 19,538 rows (run total 344,538)... next offset=550,000
DONE: /Users/yeshid/projects/seattle-care/data_proc

Continuing download to smaller files.

In [10]:
import time
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from io import StringIO

API_BASE = "https://cos-data.seattle.gov/resource/33kz-ixgy.csv"
TIME_COL = "cad_event_original_time_queued"

where = f"{TIME_COL} >= '2025-01-01T00:00:00' AND {TIME_COL} < '2026-01-01T00:00:00'"

parts_dir = Path("../data_processed/calls_2025_parts").resolve()
parts_dir.mkdir(parents=True, exist_ok=True)

LIMIT = 25_000
offset = 550_000  # <-- resume from your last printed offset
part_idx = offset // LIMIT

float_cols = [
    "care_call_sign_total_service_time_s_",
    "co_response_call_sign_total_service_time_s_",
    "spd_call_sign_total_service_time_s_",
    "call_sign_total_service_time_s_",
    "first_care_call_sign_dispatch_delay_time_s_",
    "first_care_call_sign_response_time_s_",
    "first_co_response_call_sign_dispatch_delay_time_s_",
    "first_co_response_call_sign_response_time_s_",
    "first_spd_call_sign_dispatch_delay_time_s_",
    "first_spd_call_sign_response_time_s_",
    "call_sign_response_time_s_",
    "cad_event_first_response_time_s_",
]

int_cols = ["cad_event_number", "priority", "count_of_officers"]

session = requests.Session()

def fetch_csv_page(offset: int, limit: int, max_retries: int = 6, timeout: int = 240) -> str:
    params = {"$where": where, "$limit": limit, "$offset": offset}
    last_err = None
    for attempt in range(max_retries):
        try:
            r = session.get(API_BASE, params=params, timeout=timeout)
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_err = e
            sleep_s = min(60, 2 ** attempt)
            print(f"⚠️ Fetch failed offset={offset:,} attempt={attempt+1}/{max_retries}: {e}")
            print(f"   Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
    raise last_err

total_rows = 0

while True:
    out_part = parts_dir / f"part_{part_idx:06d}.parquet"
    if out_part.exists():
        print(f"⏭️ Skipping existing {out_part.name}")
        offset += LIMIT
        part_idx += 1
        continue

    csv_text = fetch_csv_page(offset, LIMIT)
    chunk = pd.read_csv(StringIO(csv_text))

    if chunk.empty:
        print("✅ No more rows returned. Finished.")
        break

    for c in float_cols:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce").astype("float64")
    for c in int_cols:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce").astype("Int64")

    table = pa.Table.from_pandas(chunk, preserve_index=False)
    pq.write_table(table, out_part, compression="zstd")

    n = len(chunk)
    total_rows += n
    print(f"✅ Wrote {out_part.name}: {n:,} rows. Next offset={offset+LIMIT:,}")

    offset += LIMIT
    part_idx += 1

print("Done. Rows downloaded in this run:", f"{total_rows:,}")
print("Parts dir:", parts_dir)


⚠️ Fetch failed offset=550,000 attempt=1/6: HTTPSConnectionPool(host='cos-data.seattle.gov', port=443): Read timed out. (read timeout=240)
   Sleeping 1s...
✅ No more rows returned. Finished.
Done. Rows downloaded in this run: 0
Parts dir: /Users/yeshid/projects/seattle-care/data_processed/calls_2025_parts


In [15]:
#verify row counts - API count
import requests

API_COUNT = "https://cos-data.seattle.gov/resource/33kz-ixgy.json"
where = "cad_event_original_time_queued >= '2025-01-01T00:00:00' AND cad_event_original_time_queued < '2026-01-01T00:00:00'"

r = requests.get(API_COUNT, params={"$select": "count(*) as n", "$where": where}, timeout=60)
r.raise_for_status()
r.json()





[{'n': '544538'}]

In [16]:
import duckdb
from pathlib import Path

con = duckdb.connect()
full = Path("../data_processed/calls_2025_full.parquet").resolve()

have = con.execute(f"SELECT COUNT(*) FROM '{full}'").fetchone()[0]
have

344538

In [17]:
import time
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from io import StringIO

API_BASE = "https://cos-data.seattle.gov/resource/33kz-ixgy.csv"
TIME_COL = "cad_event_original_time_queued"
where = f"{TIME_COL} >= '2025-01-01T00:00:00' AND {TIME_COL} < '2026-01-01T00:00:00'"

parts_dir = Path("../data_processed/calls_2025_parts").resolve()
parts_dir.mkdir(parents=True, exist_ok=True)

LIMIT = 25_000
offset = 350_000               # <-- resume safely here
part_idx = offset // LIMIT

session = requests.Session()

def fetch_csv_page(offset: int, limit: int, max_retries: int = 8, timeout: int = 300) -> str:
    params = {"$where": where, "$limit": limit, "$offset": offset}
    last_err = None
    for attempt in range(max_retries):
        try:
            r = session.get(API_BASE, params=params, timeout=timeout)
            r.raise_for_status()
            txt = r.text
            # header-only or tiny response => retry
            if txt.strip().count("\n") < 1:
                raise RuntimeError("Got CSV header-only (likely throttled); retrying.")
            return txt
        except Exception as e:
            last_err = e
            sleep_s = min(90, 2 ** attempt)
            print(f"⚠️ Fetch failed offset={offset:,} attempt={attempt+1}/{max_retries}: {e}")
            print(f"   Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
    raise last_err

total_rows = 0
empty_streak = 0

while True:
    out_part = parts_dir / f"part_{part_idx:06d}.parquet"
    if out_part.exists():
        offset += LIMIT
        part_idx += 1
        continue

    csv_text = fetch_csv_page(offset, LIMIT)
    chunk = pd.read_csv(StringIO(csv_text))

    if chunk.empty:
        empty_streak += 1
        print(f"⚠️ Empty chunk at offset={offset:,} (empty_streak={empty_streak})")
        if empty_streak >= 2:
            print("✅ Two consecutive empty pages → finished.")
            break
        time.sleep(5)
        continue

    empty_streak = 0

    # stabilize a couple known-problem types
    if "call_sign_total_service_time_s_" in chunk.columns:
        chunk["call_sign_total_service_time_s_"] = pd.to_numeric(
            chunk["call_sign_total_service_time_s_"], errors="coerce"
        ).astype("float64")
    if "priority" in chunk.columns:
        chunk["priority"] = pd.to_numeric(chunk["priority"], errors="coerce").astype("Int64")

    table = pa.Table.from_pandas(chunk, preserve_index=False)
    pq.write_table(table, out_part, compression="zstd")

    n = len(chunk)
    total_rows += n
    print(f"✅ Wrote {out_part.name}: {n:,} rows. Next offset={offset+LIMIT:,}")

    offset += LIMIT
    part_idx += 1
    time.sleep(0.4)  # gentle throttle

print("Done. Rows downloaded in this run:", f"{total_rows:,}")
print("Parts dir:", parts_dir)


⚠️ Fetch failed offset=350,000 attempt=1/8: HTTPSConnectionPool(host='cos-data.seattle.gov', port=443): Read timed out. (read timeout=300)
   Sleeping 1s...
✅ Wrote part_000014.parquet: 25,000 rows. Next offset=375,000
✅ Wrote part_000015.parquet: 25,000 rows. Next offset=400,000
✅ Wrote part_000016.parquet: 25,000 rows. Next offset=425,000
✅ Wrote part_000017.parquet: 25,000 rows. Next offset=450,000
✅ Wrote part_000018.parquet: 25,000 rows. Next offset=475,000
✅ Wrote part_000019.parquet: 25,000 rows. Next offset=500,000
✅ Wrote part_000020.parquet: 25,000 rows. Next offset=525,000
✅ Wrote part_000021.parquet: 19,538 rows. Next offset=550,000
⚠️ Fetch failed offset=550,000 attempt=1/8: Got CSV header-only (likely throttled); retrying.
   Sleeping 1s...
⚠️ Fetch failed offset=550,000 attempt=2/8: Got CSV header-only (likely throttled); retrying.
   Sleeping 2s...
⚠️ Fetch failed offset=550,000 attempt=3/8: Got CSV header-only (likely throttled); retrying.
   Sleeping 4s...
⚠️ Fetch fa

RuntimeError: Got CSV header-only (likely throttled); retrying.

In [19]:
#combine the parquet files. all data download complete.

import duckdb
from pathlib import Path

con = duckdb.connect()

full = Path("../data_processed/calls_2025_full.parquet").resolve()
parts_glob = str(Path("../data_processed/calls_2025_parts/part_*.parquet").resolve())
final_out = Path("../data_processed/calls_2025_all.parquet").resolve()

# Combine and deduplicate by cad_event_number (keep the earliest queued time per event)
con.execute(f"""
COPY (
  SELECT * EXCLUDE (row_num)
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY cad_event_number
             ORDER BY cad_event_original_time_queued
           ) AS row_num
    FROM (
      SELECT * FROM '{full}'
      UNION ALL
      SELECT * FROM '{parts_glob}'
    )
  )
  WHERE row_num = 1
)
TO '{final_out}' (FORMAT PARQUET);
""")

con.execute(f"SELECT COUNT(*) AS n FROM '{final_out}'").df()



,n
0,205715


More checks on data downloaded.

In [20]:
con.execute(f"""
SELECT
  MIN(cad_event_original_time_queued) AS min_time,
  MAX(cad_event_original_time_queued) AS max_time
FROM '{final_out}'
""").df()


,min_time,max_time
0,2025-05-15T05:16:59.000,2025-12-12T23:59:44.000


In [21]:
from pathlib import Path
import duckdb

con = duckdb.connect()
final_out = Path("../data_processed/calls_2025_all.parquet").resolve()

con.execute(f"SELECT COUNT(*) AS n FROM '{final_out}'").df()


,n
0,205715


In [22]:
from pathlib import Path

base = Path("../data_processed").resolve()

print("Full file exists:", (base / "calls_2025_full.parquet").exists())
print("Full file size (MB):", (base / "calls_2025_full.parquet").stat().st_size / 1e6)

parts_dir = base / "calls_2025_parts"
parts = sorted(parts_dir.glob("part_*.parquet"))

print("Number of part files:", len(parts))
for p in parts:
    print(p.name)


Full file exists: True
Full file size (MB): 26.507164
Number of part files: 8
part_000014.parquet
part_000015.parquet
part_000016.parquet
part_000017.parquet
part_000018.parquet
part_000019.parquet
part_000020.parquet
part_000021.parquet


In [23]:
import duckdb
con = duckdb.connect()

# Count rows in the initial full file
n_full = con.execute("""
SELECT COUNT(*) FROM '../data_processed/calls_2025_full.parquet'
""").fetchone()[0]

# Count rows across all parts
n_parts = con.execute("""
SELECT COUNT(*) FROM '../data_processed/calls_2025_parts/part_*.parquet'
""").fetchone()[0]

n_full, n_parts, n_full + n_parts


(344538, 194538, 539076)

Adding final data chunk that didnt get download.

In [24]:
import time
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from io import StringIO

API_BASE = "https://cos-data.seattle.gov/resource/33kz-ixgy.csv"
TIME_COL = "cad_event_original_time_queued"
where = f"{TIME_COL} >= '2025-01-01T00:00:00' AND {TIME_COL} < '2026-01-01T00:00:00'"

parts_dir = Path("../data_processed/calls_2025_parts").resolve()
parts_dir.mkdir(parents=True, exist_ok=True)

LIMIT = 25_000
offset = 325_000
part_idx = offset // LIMIT

out_part = parts_dir / f"part_{part_idx:06d}.parquet"
print("Will write:", out_part)

session = requests.Session()

def fetch_csv_page(offset: int, limit: int, max_retries: int = 8, timeout: int = 300) -> str:
    params = {"$where": where, "$limit": limit, "$offset": offset}
    last_err = None
    for attempt in range(max_retries):
        try:
            r = session.get(API_BASE, params=params, timeout=timeout)
            r.raise_for_status()
            txt = r.text
            if txt.strip().count("\n") < 1:
                raise RuntimeError("Got CSV header-only; retrying.")
            return txt
        except Exception as e:
            last_err = e
            sleep_s = min(60, 2 ** attempt)
            print(f"⚠️ Fetch failed offset={offset:,} attempt={attempt+1}/{max_retries}: {e}")
            print(f"   Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
    raise last_err

csv_text = fetch_csv_page(offset, LIMIT)
chunk = pd.read_csv(StringIO(csv_text))

# stabilize a couple known-problem types
if "call_sign_total_service_time_s_" in chunk.columns:
    chunk["call_sign_total_service_time_s_"] = pd.to_numeric(chunk["call_sign_total_service_time_s_"], errors="coerce").astype("float64")
if "priority" in chunk.columns:
    chunk["priority"] = pd.to_numeric(chunk["priority"], errors="coerce").astype("Int64")

table = pa.Table.from_pandas(chunk, preserve_index=False)
pq.write_table(table, out_part, compression="zstd")

len(chunk), out_part


Will write: /Users/yeshid/projects/seattle-care/data_processed/calls_2025_parts/part_000013.parquet


(25000,
 PosixPath('/Users/yeshid/projects/seattle-care/data_processed/calls_2025_parts/part_000013.parquet'))

In [25]:
import duckdb
from pathlib import Path

con = duckdb.connect()

full = Path("../data_processed/calls_2025_full.parquet").resolve()
parts_glob = str(Path("../data_processed/calls_2025_parts/part_*.parquet").resolve())
final_out = Path("../data_processed/calls_2025_all.parquet").resolve()

con.execute(f"""
COPY (
  SELECT * EXCLUDE (row_num)
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY cad_event_number
             ORDER BY cad_event_original_time_queued
           ) AS row_num
    FROM (
      SELECT * FROM '{full}'
      UNION ALL
      SELECT * FROM read_parquet('{parts_glob}')
    )
  )
  WHERE row_num = 1
)
TO '{final_out}' (FORMAT PARQUET);
""")

con.execute(f"SELECT COUNT(*) AS n FROM '{final_out}'").df()


,n
0,205715


In [27]:
#checking union
import duckdb
con = duckdb.connect()

n_full = con.execute("SELECT COUNT(*) FROM '../data_processed/calls_2025_full.parquet'").fetchone()[0]
n_parts = con.execute("SELECT COUNT(*) FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')").fetchone()[0]

n_union = con.execute("""
SELECT COUNT(*) 
FROM (
  SELECT * FROM '../data_processed/calls_2025_full.parquet'
  UNION ALL
  SELECT * FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')
)
""").fetchone()[0]

(n_full, n_parts, n_union, n_full + n_parts)


(344538, 219538, 564076, 564076)

In [29]:
con.execute("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(cad_event_number) AS n_nonnull_cad_event_number,
  COUNT(*) - COUNT(cad_event_number) AS n_null_cad_event_number,
  COUNT(DISTINCT cad_event_number) AS n_distinct_cad_event_number
FROM (
  SELECT * FROM '../data_processed/calls_2025_full.parquet'
  UNION ALL
  SELECT * FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')
)
""").df()


,n_rows,n_nonnull_cad_event_number,n_null_cad_event_number,n_distinct_cad_event_number
0,564076,564076,0,205715


In [31]:
#dedup at row level not cad level

import duckdb
from pathlib import Path

con = duckdb.connect()

final_out = Path("../data_processed/calls_2025_all.parquet").resolve()

con.execute(f"""
COPY (
  SELECT DISTINCT *
  FROM (
    SELECT * FROM '../data_processed/calls_2025_full.parquet'
    UNION ALL
    SELECT * FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')
  )
)
TO '{final_out}' (FORMAT PARQUET);
""")

con.execute(f"SELECT COUNT(*) AS n FROM '{final_out}'").df()


,n
0,344538


In [32]:
con.execute(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT cad_event_number) AS n_events
FROM '{final_out}'
""").df()


,n_rows,n_events
0,344538,205715


In [40]:
import duckdb
import pandas as pd
import os
import io

# Define the output path variables early
output_dir = '../data_processed/'
filename = 'cad_duplication_distribution_2025.csv'
output_path = os.path.join(output_dir, filename)

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# 1. Connect to DuckDB
con = duckdb.connect()

# Define the combined data source as a CTE (adjust file paths if necessary)
RAW_CALLS_CTE = """
(
  -- NOTE: Ensure your file paths are correct!
  SELECT * FROM '../data_processed/calls_2025_full.parquet'
  UNION ALL
  SELECT * FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')
)
"""

# --- QUERY 1: Find the MOST Duplicated CAD Event Number ---
highest_duplication_query = f"""
SELECT 
  cad_event_number, 
  COUNT(*) AS n_duplicates
FROM 
  {RAW_CALLS_CTE}
GROUP BY 
  cad_event_number
ORDER BY 
  n_duplicates DESC
LIMIT 1;
"""

df_max_duplication = con.execute(highest_duplication_query).df()
print("--- 1. Most Duplicated CAD Event ---")
if not df_max_duplication.empty:
    target_cad_id = df_max_duplication['cad_event_number'].iloc[0]
    max_count = df_max_duplication['n_duplicates'].iloc[0]
    print(f"CAD Event Number: {target_cad_id}")
    print(f"Total Records: {max_count}\n")
else:
    print("No data found.")
    con.close()
    # Skip Query 2 and saving if no data is found
    # return

# --- QUERY 2: Find the General Duplication Distribution ---
duplication_distribution_query = f"""
WITH DuplicationCounts AS (
    SELECT
        cad_event_number,
        COUNT(*) AS n_duplicates
    FROM
        {RAW_CALLS_CTE}
    GROUP BY
        cad_event_number
)
SELECT
    n_duplicates,
    COUNT(*) AS n_cad_events_at_this_level,
    -- Calculate the percentage of all unique events that fall into this duplication count
    (COUNT(*) * 100.0) / (SELECT COUNT(DISTINCT cad_event_number) FROM {RAW_CALLS_CTE}) AS percent_of_total_events
FROM
    DuplicationCounts
GROUP BY
    n_duplicates
ORDER BY
    n_duplicates DESC;
"""

df_distribution = con.execute(duplication_distribution_query).df()
con.close()

# --- Save the Distribution Data ---
df_distribution.to_csv(output_path, index=False)

print(f"✅ Duplication distribution saved successfully to: {output_path}")

print("\n--- 2. General Duplication Distribution ---")
# Display the distribution
print(df_distribution)

--- 1. Most Duplicated CAD Event ---
CAD Event Number: 2025000353113
Total Records: 200

✅ Duplication distribution saved successfully to: ../data_processed/cad_duplication_distribution_2025.csv

--- 2. General Duplication Distribution ---
    n_duplicates  n_cad_events_at_this_level  percent_of_total_events
0            200                           1                 0.000486
1            114                           1                 0.000486
2            104                           1                 0.000486
3            102                           1                 0.000486
4             98                           1                 0.000486
..           ...                         ...                      ...
62             5                        1257                 0.611040
63             4                       24400                11.861070
64             3                        5935                 2.885059
65             2                       99574                

results from below code: sanity check done on duplication distribution.
--- Sanity Check Results ---
Total Unique CAD Events (Expected: 205,715): 205715
Total Log Entries (Rows) Calculated from Distribution (Expected: 564,076): 564076

--- Verification Status ---
✅ SUCCESS: The calculated totals from the distribution exactly match the original raw row counts.

In [41]:
import pandas as pd
import os
import numpy as np

# Define the path where the distribution data was saved
output_dir = '../data_processed/'
filename = 'cad_duplication_distribution_2025.csv'
output_path = os.path.join(output_dir, filename)

try:
    # Check if the directory exists and list files for debugging
    if not os.path.exists(output_dir):
        print(f"Directory {output_dir} does not exist.")
    # Attempt to load the distribution data
    df_distribution = pd.read_csv(output_path)
    
    # 1. Calculate the total number of unique CAD events
    total_unique_events = df_distribution['n_cad_events_at_this_level'].sum()
    
    # 2. Calculate the total number of rows (log entries) from the distribution
    df_distribution['total_rows_at_level'] = df_distribution['n_duplicates'] * df_distribution['n_cad_events_at_this_level']
    total_log_entries_from_distribution = df_distribution['total_rows_at_level'].sum()
    
    # --- Sanity Check Logic ---
    print("--- Sanity Check Results ---")
    print(f"Total Unique CAD Events (Expected: 205,715): {total_unique_events}")
    print(f"Total Log Entries (Rows) Calculated from Distribution (Expected: 564,076): {total_log_entries_from_distribution}")
    
    # Verification Status
    expected_unique = 205715
    expected_total_rows = 564076
    
    unique_match = (total_unique_events == expected_unique)
    total_rows_match = (total_log_entries_from_distribution == expected_total_rows)
    
    print("\n--- Verification Status ---")
    if unique_match and total_rows_match:
        print("✅ SUCCESS: The calculated totals from the distribution exactly match the original raw row counts.")
    else:
        print("❌ MISMATCH DETECTED: The distribution does not perfectly reconstruct the original counts. Review the original DuckDB query and file saving steps.")
        print(f"Unique Events Match: {unique_match}")
        print(f"Total Rows Match: {total_rows_match}")
        
except FileNotFoundError:
    # If the file is still not found, we need to locate it manually or re-save it.
    print(f"❌ Critical Error: The file {output_path} still cannot be found. Please check your file system.")
except Exception as e:
    print(f"❌ An unexpected error occurred during processing: {e}")

--- Sanity Check Results ---
Total Unique CAD Events (Expected: 205,715): 205715
Total Log Entries (Rows) Calculated from Distribution (Expected: 564,076): 564076

--- Verification Status ---
✅ SUCCESS: The calculated totals from the distribution exactly match the original raw row counts.


In [45]:
import duckdb
import pandas as pd

con = duckdb.connect()

# The DESCRIBE command is the 'gold standard' for inspecting files
parquet_file = "../data_processed/calls_2025_full.parquet"
query = f"DESCRIBE SELECT * FROM '{parquet_file}' LIMIT 0;"

df_header = con.execute(query).df()

print("--- FULL TABLE HEADER (SCHEMA) ---")
# 'column_name' and 'column_type' are the standard output columns for DESCRIBE
print(df_header[['column_name', 'column_type']])

con.close()

--- FULL TABLE HEADER (SCHEMA) ---
                                          column_name column_type
0                                    cad_event_number      BIGINT
1                     cad_event_clearance_description     VARCHAR
2                                           call_type     VARCHAR
3                                            priority      BIGINT
4                                   initial_call_type     VARCHAR
5                                     final_call_type     VARCHAR
6                      cad_event_original_time_queued     VARCHAR
7                              cad_event_arrived_time     VARCHAR
8                                   dispatch_precinct     VARCHAR
9                                     dispatch_sector     VARCHAR
10                                      dispatch_beat     VARCHAR
11                                 dispatch_longitude     VARCHAR
12                                  dispatch_latitude     VARCHAR
13                            dispatch_re

variance and duplication type check.

In [50]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Define the combined source
RAW_CALLS_VIEW = """
(
  SELECT * FROM '../data_processed/calls_2025_full.parquet'
  UNION ALL
  SELECT * FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')
)
"""

# The specific ID we found earlier
target_id = '2025000353113'

# We select the columns most likely to show the progression of the call
query = f"""
SELECT 
    cad_event_number,
    cad_event_original_time_queued,
    cad_event_arrived_time,
    -- We use '*' to see if there are other timestamp columns if these names are slightly off
    initial_call_type,
    final_call_type
FROM {RAW_CALLS_VIEW}
WHERE cad_event_number = '{target_id}'
ORDER BY cad_event_original_time_queued ASC, cad_event_arrived_time ASC
"""

df_inspect = con.execute(query).df()

# Print the first 10 and last 10 rows to see the start and end of the event
print(f"--- Full History for CAD: {target_id} ---")
print(df_inspect.head(10))
print("\n... intermediate rows ...\n")
print(df_inspect.tail(10))


# This query checks for variance within the 200 rows of this specific event
query_variance = f"""
SELECT 
    COUNT(DISTINCT cad_event_original_time_queued) as unique_start_times,
    COUNT(DISTINCT cad_event_arrived_time) as unique_arrival_times,
    COUNT(DISTINCT initial_call_type) as unique_initial_types,
    COUNT(DISTINCT final_call_type) as unique_final_types
FROM {RAW_CALLS_VIEW}
WHERE cad_event_number = '{target_id}'
"""

df_variance = con.execute(query_variance).df()

print(f"--- Variance Analysis for CAD: {target_id} ---")
print(df_variance)

con.close()

--- Full History for CAD: 2025000353113 ---
   cad_event_number cad_event_original_time_queued   cad_event_arrived_time  \
0     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
1     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
2     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
3     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
4     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
5     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
6     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
7     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
8     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   
9     2025000353113        2025-12-02T13:31:55.000  2025-12-02T13:31:55.000   

               initial_call_type  final_call_type  
0  ASLT - PERSON SHOT OR SHOT AT  

In [51]:
#dedup to distinct rows
import duckdb
import pandas as pd
import os

con = duckdb.connect()

# Combine sources
RAW_CALLS_VIEW = """
(
  SELECT * FROM '../data_processed/calls_2025_full.parquet'
  UNION ALL
  SELECT * FROM read_parquet('../data_processed/calls_2025_parts/part_*.parquet')
)
"""

# DISTINCT * collapses rows only if EVERY column is an exact match
dedup_query = f"""
SELECT DISTINCT * FROM {RAW_CALLS_VIEW}
"""

print("Cleaning data... collapsing identical duplicates.")
df_clean = con.execute(dedup_query).df()

# SAVE the clean version immediately
output_path = '../data_processed/calls_2025_clean_unique.parquet'
df_clean.to_parquet(output_path, index=False)

print(f"✅ Clean dataset saved to: {output_path}")
print(f"Original Row Count: ~564,000")
print(f"Cleaned Row Count: {len(df_clean)}")

con.close()

Cleaning data... collapsing identical duplicates.
✅ Clean dataset saved to: ../data_processed/calls_2025_clean_unique.parquet
Original Row Count: ~564,000
Cleaned Row Count: 344538


If your count dropped from 564k to 344k, but your unique ID count is still 205k, it means:

Partial Duplication: About 220,000 rows were exact, 100% identical "ghost" duplicates (these were removed).

True Activity Logs: There are still about 138,823 rows (344,538 minus 205,715) that have the same CAD ID but differ in at least one column.

In [54]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Use the CLEANED file you just saved
CLEAN_DATA = "../data_processed/calls_2025_clean_unique.parquet"

target_id = '2025000353113'

# Check variance again on the CLEANED data
query = f"""
SELECT 
    COUNT(*) as total_rows_remaining,
    COUNT(DISTINCT cad_event_arrived_time) as unique_arrival_times
FROM '{CLEAN_DATA}'
WHERE cad_event_number = '{target_id}'
"""

df_check = con.execute(query).df()
con.close()

print(f"--- Analysis of REMAINING rows for CAD: {target_id} ---")
print(df_check)

--- Analysis of REMAINING rows for CAD: 2025000353113 ---
   total_rows_remaining  unique_arrival_times
0                   100                     1
